In [ ]:
# =========================
# CAMEMBERT FEATURES EXTRACTION AVANCÉ
# =========================
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from transformers import CamembertTokenizer, CamembertModel, AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("CAMEMBERT FEATURES EXTRACTION AVANCÉ")
print("="*60)

# =========================
# 1. CONFIGURATION
# =========================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

BATCH_SIZE = 16 if device.type == 'cuda' else 8
MAX_LENGTH = 128
print(f"Batch size: {BATCH_SIZE}")
print(f"Max length: {MAX_LENGTH}")

# =========================
# 2. CHARGEMENT DES DONNÉES
# =========================
print("Chargement des textes...")
train_texts = pd.read_csv('/kaggle/input/data-5/train_texts (2).csv')
test_texts = pd.read_csv('/kaggle/input/data-5/test_texts (2).csv')
y_train = pd.read_csv('/kaggle/input/data-5/y_train (2).csv')['label'].values
test_ids = pd.read_csv('/kaggle/input/data-5/test_ids (2).csv')['challenge_id'].values

print(f"Train: {len(train_texts)} textes")
print(f"Test: {len(test_texts)} textes")

# =========================
# 3. CHARGEMENT DU MODÈLE
# =========================
print("Chargement de CamemBERT...")

try:
    # Essayer CamemBERT-base d'abord
    tokenizer = CamembertTokenizer.from_pretrained('camembert-base')
    model = CamembertModel.from_pretrained('camembert-base')
    print("CamemBERT-base chargé")
except Exception as e:
    print(f"Erreur CamemBERT-base: {e}")
    print("Tentative avec camembert/camembert-base...")
    try:
        tokenizer = AutoTokenizer.from_pretrained('camembert/camembert-base')
        model = AutoModel.from_pretrained('camembert/camembert-base')
        print("camembert/camembert-base chargé")
    except:
        print("Impossible de charger CamemBERT")
        exit()

model.to(device)
model.eval()

# =========================
# 4. DATASET PYTORCH
# =========================
class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx]['full_text'])
        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0)
        }

# =========================
# 5. EXTRACTION DES EMBEDDINGS
# =========================
def extract_embeddings_advanced(texts, tokenizer, model, batch_size, max_length):
    """
    Extrait plusieurs types d'embeddings:
    - CLS token (représentation globale)
    - Mean pooling (moyenne de tous les tokens)
    - Max pooling (maximum de tous les tokens)
    - Last 4 layers concatenated (plus riche)
    """
    dataset = TextDataset(texts, tokenizer, max_length)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    all_cls = []
    all_mean = []
    all_max = []
    
    print("\n Extraction des embeddings...")
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Processing batches"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            # Forward pass
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )
            
            # CLS token (premier token) de la dernière couche
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_cls.append(cls_embeddings)
            
            # Mean pooling (moyenne sur les tokens, en excluant le padding)
            attention_mask_expanded = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
            sum_embeddings = torch.sum(outputs.last_hidden_state * attention_mask_expanded, dim=1)
            sum_mask = torch.clamp(attention_mask_expanded.sum(dim=1), min=1e-9)
            mean_embeddings = (sum_embeddings / sum_mask).cpu().numpy()
            all_mean.append(mean_embeddings)
            
            # Max pooling
            max_embeddings = torch.max(outputs.last_hidden_state * attention_mask_expanded, dim=1)[0].cpu().numpy()
            all_max.append(max_embeddings)
    
    # Concatener tous les batches
    cls_embeddings = np.vstack(all_cls)
    mean_embeddings = np.vstack(all_mean)
    max_embeddings = np.vstack(all_max)
    
    print(f"CLS embeddings: {cls_embeddings.shape}")
    print(f"Mean embeddings: {mean_embeddings.shape}")
    print(f"Max embeddings: {max_embeddings.shape}")
    
    return cls_embeddings, mean_embeddings, max_embeddings

# =========================
# 6. EXTRACTION TRAIN
# =========================
print("\n" + "="*60)
print("EXTRACTION TRAIN")
print("="*60)

train_cls, train_mean, train_max = extract_embeddings_advanced(
    train_texts, tokenizer, model, BATCH_SIZE, MAX_LENGTH
)

# =========================
# 7. EXTRACTION TEST
# =========================
print("\n" + "="*60)
print("EXTRACTION TEST")
print("="*60)

test_cls, test_mean, test_max = extract_embeddings_advanced(
    test_texts, tokenizer, model, BATCH_SIZE, MAX_LENGTH
)

# =========================
# 8. COMBINAISON DES EMBEDDINGS
# =========================
print("\n Combinaison des différents embeddings...")

# Stratégie 1: Concatener CLS + Mean + Max
train_embeddings_full = np.hstack([train_cls, train_mean, train_max])
test_embeddings_full = np.hstack([test_cls, test_mean, test_max])

print(f"Embeddings complets: {train_embeddings_full.shape}")

# Stratégie 2: Juste CLS (plus simple)
train_embeddings_cls = train_cls
test_embeddings_cls = test_cls

# Stratégie 3: Mean pooling seul
train_embeddings_mean = train_mean
test_embeddings_mean = test_mean

# =========================
# 9. RÉDUCTION DE DIMENSIONNALITÉ
# =========================
print("\n Réduction de dimensionnalité...")

# SVD pour les embeddings complets (768*3 = 2304 -> 150)
print("\n1️⃣  SVD sur embeddings complets (2304 -> 150)...")
svd_full = TruncatedSVD(n_components=150, random_state=42)
train_reduced_full = svd_full.fit_transform(train_embeddings_full)
test_reduced_full = svd_full.transform(test_embeddings_full)
print(f"   Variance expliquée: {svd_full.explained_variance_ratio_.sum():.2%}")

# SVD pour CLS seul (768 -> 100)
print("\n2️⃣  SVD sur CLS (768 -> 100)...")
svd_cls = TruncatedSVD(n_components=100, random_state=42)
train_reduced_cls = svd_cls.fit_transform(train_embeddings_cls)
test_reduced_cls = svd_cls.transform(test_embeddings_cls)
print(f"   Variance expliquée: {svd_cls.explained_variance_ratio_.sum():.2%}")

# SVD pour Mean (768 -> 100)
print("\n3️⃣  SVD sur Mean Pooling (768 -> 100)...")
svd_mean = TruncatedSVD(n_components=100, random_state=42)
train_reduced_mean = svd_mean.fit_transform(train_embeddings_mean)
test_reduced_mean = svd_mean.transform(test_embeddings_mean)
print(f"   Variance expliquée: {svd_mean.explained_variance_ratio_.sum():.2%}")

# =========================
# 10. FEATURES STATISTIQUES SUR LES EMBEDDINGS
# =========================
print("\n Extraction de features statistiques...")

def extract_statistical_features(embeddings):
    """Extrait des features statistiques des embeddings"""
    features = {}
    
    # Statistiques de base
    features['mean'] = embeddings.mean(axis=1, keepdims=True)
    features['std'] = embeddings.std(axis=1, keepdims=True)
    features['min'] = embeddings.min(axis=1, keepdims=True)
    features['max'] = embeddings.max(axis=1, keepdims=True)
    features['median'] = np.median(embeddings, axis=1, keepdims=True)
    
    # Percentiles
    features['q25'] = np.percentile(embeddings, 25, axis=1, keepdims=True)
    features['q75'] = np.percentile(embeddings, 75, axis=1, keepdims=True)
    
    # Range et IQR
    features['range'] = features['max'] - features['min']
    features['iqr'] = features['q75'] - features['q25']
    
    # Skewness et kurtosis (approximations)
    from scipy.stats import skew, kurtosis
    features['skew'] = skew(embeddings, axis=1).reshape(-1, 1)
    features['kurtosis'] = kurtosis(embeddings, axis=1).reshape(-1, 1)
    
    # Concatener toutes les features
    all_features = np.hstack(list(features.values()))
    return all_features

train_stats = extract_statistical_features(train_embeddings_cls)
test_stats = extract_statistical_features(test_embeddings_cls)
print(f"Features statistiques: {train_stats.shape}")

# =========================
# 11. NORMALISATION
# =========================
print("\n Normalisation des features...")

scaler = StandardScaler()

# Normaliser les réductions SVD
train_reduced_full_scaled = scaler.fit_transform(train_reduced_full)
test_reduced_full_scaled = scaler.transform(test_reduced_full)

train_reduced_cls_scaled = scaler.fit_transform(train_reduced_cls)
test_reduced_cls_scaled = scaler.transform(test_reduced_cls)

print("Normalisation terminée")

# =========================
# 12. SAUVEGARDE
# =========================
print("\n Sauvegarde des embeddings...")

# Sauvegarder les embeddings bruts complets (pour backup)
np.save('train_camembert_full.npy', train_embeddings_full)
np.save('test_camembert_full.npy', test_embeddings_full)
np.save('train_camembert_cls.npy', train_embeddings_cls)
np.save('test_camembert_cls.npy', test_embeddings_cls)

# Sauvegarder les réductions (pour les modèles)
np.save('train_camembert_reduced_150.npy', train_reduced_full)
np.save('test_camembert_reduced_150.npy', test_reduced_full)
np.save('train_camembert_reduced_100.npy', train_reduced_cls)
np.save('test_camembert_reduced_100.npy', test_reduced_cls)

# Sauvegarder les versions normalisées
np.save('train_camembert_reduced_150_scaled.npy', train_reduced_full_scaled)
np.save('test_camembert_reduced_150_scaled.npy', test_reduced_full_scaled)

# Sauvegarder les features statistiques
np.save('train_camembert_stats.npy', train_stats)
np.save('test_camembert_stats.npy', test_stats)

# Créer des DataFrames CSV pour usage facile
print("\n Création des fichiers CSV...")

# Version 1: Réduction 150 dims (meilleure qualité)
train_df_150 = pd.DataFrame(
    train_reduced_full_scaled,
    columns=[f'camembert_150_{i}' for i in range(150)]
)
test_df_150 = pd.DataFrame(
    test_reduced_full_scaled,
    columns=[f'camembert_150_{i}' for i in range(150)]
)
train_df_150.to_csv('train_camembert_features_150.csv', index=False)
test_df_150.to_csv('test_camembert_features_150.csv', index=False)

# Version 2: Réduction 100 dims (plus léger)
train_df_100 = pd.DataFrame(
    train_reduced_cls_scaled,
    columns=[f'camembert_100_{i}' for i in range(100)]
)
test_df_100 = pd.DataFrame(
    test_reduced_cls_scaled,
    columns=[f'camembert_100_{i}' for i in range(100)]
)
train_df_100.to_csv('train_camembert_features_100.csv', index=False)
test_df_100.to_csv('test_camembert_features_100.csv', index=False)

# Version 3: Features statistiques
train_df_stats = pd.DataFrame(
    train_stats,
    columns=[f'camembert_stat_{i}' for i in range(train_stats.shape[1])]
)
test_df_stats = pd.DataFrame(
    test_stats,
    columns=[f'camembert_stat_{i}' for i in range(test_stats.shape[1])]
)
train_df_stats.to_csv('train_camembert_stats.csv', index=False)
test_df_stats.to_csv('test_camembert_stats.csv', index=False)

print("\n Fichiers sauvegardés:")
print("\n Embeddings complets (.npy):")
print("  - train_camembert_full.npy (768*3 dims)")
print("  - test_camembert_full.npy (768*3 dims)")
print("  - train_camembert_cls.npy (768 dims)")
print("  - test_camembert_cls.npy (768 dims)")

print("\n Réductions SVD (.npy):")
print("  - train_camembert_reduced_150.npy")
print("  - test_camembert_reduced_150.npy")
print("  - train_camembert_reduced_100.npy")
print("  - test_camembert_reduced_100.npy")

print("\n Features CSV:")
print("  - train_camembert_features_150.csv (150 dims, RECOMMANDÉ)")
print("  - test_camembert_features_150.csv")
print("  - train_camembert_features_100.csv (100 dims)")
print("  - test_camembert_features_100.csv")
print("  - train_camembert_stats.csv (features statistiques)")
print("  - test_camembert_stats.csv")

# =========================
# 13. TEST RAPIDE
# =========================
print("\n Test rapide des features CamemBERT...")

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Test avec version 150
lr = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
scores_150 = cross_val_score(lr, train_reduced_full_scaled, y_train, cv=5, scoring='accuracy')
print(f"\n CamemBERT 150 dims - CV Accuracy: {scores_150.mean():.4f} (+/- {scores_150.std():.4f})")

# Test avec version 100
scores_100 = cross_val_score(lr, train_reduced_cls_scaled, y_train, cv=5, scoring='accuracy')
print(f"CamemBERT 100 dims - CV Accuracy: {scores_100.mean():.4f} (+/- {scores_100.std():.4f})")

# Test avec stats
scores_stats = cross_val_score(lr, train_stats, y_train, cv=5, scoring='accuracy')
print(f"CamemBERT stats - CV Accuracy: {scores_stats.mean():.4f} (+/- {scores_stats.std():.4f})")

# Combiner stats + reduced
combined = np.hstack([train_reduced_full_scaled, train_stats])
scores_combined = cross_val_score(lr, combined, y_train, cv=5, scoring='accuracy')
print(f"CamemBERT combined - CV Accuracy: {scores_combined.mean():.4f} (+/- {scores_combined.std():.4f})")



print("\n EXTRACTION CAMEMBERT TERMINÉE")
print("="*60)
print("\n  Prochaine étape: Intégrer ces features dans l'ensemble final")